# 10 · Crop-signal — 04a center-crop có vứt bỏ thông tin phục vụ nhãn không?

`08/B.4` đo: 04a scale cạnh ngắn → center-crop vuông ⇒ **cắt trung bình 15% trục dài**
(p50 16.8%, p99 25.2%). Phim dọc mất đỉnh phổi + góc sườn hoành; phim ngang mất thành ngực.
Câu hỏi: **vùng bị cắt có mang tín hiệu không?**

## Phép đo — multi-crop CÙNG PHÂN PHỐI (tránh confound méo/pad)

Từ ảnh **gốc** (Drive), scale cạnh ngắn tới 559 (như 04a) → crop 559×559 ở **ba vị trí dọc
trục dài**:

| crop | vị trí | = gì |
|---|---|---|
| `ctr` | giữa | **đúng pipeline hiện tại** |
| `lo` | đầu (top/left-aligned) | vùng 04a cắt bỏ một đầu |
| `hi` | cuối (bottom/right-aligned) | vùng 04a cắt bỏ đầu kia |

Cả ba cùng scale, cùng 559, cùng phân phối pretrain → chỉ khác **vị trí**. RAD-DINO (DINOv2)
pretrain đa-crop nên bất biến vị trí ⇒ ba crop đều in-distribution. **Ảnh vuông**: ba vị trí
trùng nhau ⇒ tự động đóng góp 0, không cần lọc.

## So sánh (linear probe, mức study 7:3 — giống 08)

- `ctr` (hiện tại)  vs  `mean(ctr,lo,hi)` (TTA đa-crop)  vs  `concat` (probe thấy cả ba)
- Nếu `mean`/`concat` > `ctr` **quá +0.005** ⇒ vùng bị cắt có tín hiệu ⇒ đáng cân nhắc
  đa-crop inference hoặc khung rộng hơn.
- **Phân tầng theo lượng cắt**: hiệu ứng phải tập trung ở ảnh bị cắt nhiều, không ở ảnh vuông.

> ⚠️ Subsample phân tầng (oversample nhãn hiếm) ⇒ mAP tuyệt đối KHÔNG so với 0.4336 của H5.
> Chỉ **hiệu số giữa các crop** có nghĩa. Đây là **screen** (probe + 1 seed) — nếu vượt
> ngưỡng mới cân nhắc kiểm bằng ML-Decoder + CI.


## C0 · Config

In [1]:
import os, json, math, io, pickle, random
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from PIL import Image
from tqdm.auto import tqdm

try:
    from google.colab import drive
    drive.mount('/content/drive'); DATA_DIR = '/content/drive/MyDrive/MIMIC_CXR/planB_out'
except Exception:
    DATA_DIR = '.'
OUT_DIR   = os.path.join(DATA_DIR, 'cohort_v2')
DRIVE_IMG = '/content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files'  # anh GOC
BB_CKPT   = os.path.join(DATA_DIR, 'lpft_v2', 'raddino_mld_db_ft_best.pt')
CS_DIR    = os.path.join(DATA_DIR, 'crop_signal'); os.makedirs(CS_DIR, exist_ok=True)

MIN_POS      = 250     # ca duong toi thieu MOI nhan, moi tap (screen -> nho hon 08)
N_BACKGROUND = 2500
CROP_HEADROOM = 1.08   # giong 04a
W_F, W_L = 0.7, 0.3
SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
dev = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', dev, '|', torch.cuda.get_device_name(0) if dev=='cuda' else 'CPU')

PAL = {'macro':'#0072B2', 'head':'#D55E00', 'medium':'#009E73', 'tail':'#7B3294'}

Mounted at /content/drive
device: cuda | NVIDIA RTX PRO 6000 Blackwell Server Edition


## C1 · Cohort + subsample phân tầng + kiểm ảnh gốc trên Drive

`view_map` giữ `subject_id` ⇒ dựng path Drive `p{sub[:2]}/p{sub}/s{study}/{dicom}.jpg`.
**Ảnh gốc (615 GB) phải còn trên Drive** — 04a đã dùng chúng. Cell assert một ảnh mẫu load được.

In [2]:
meta   = json.load(open(os.path.join(OUT_DIR,'cohort_meta.json')))
LABELS = meta['labels']; assert len(LABELS)==13
ehr = pd.read_parquet(os.path.join(OUT_DIR,'cohort_study_level.parquet')).sort_values('study_id').reset_index(drop=True)
cf = pickle.load(open(os.path.join(OUT_DIR,'class_freq.pkl'),'rb'))
TRAIN_PREV = np.asarray(cf['class_freq'])/cf['num_train']
TIER = np.where(TRAIN_PREV<0.02,'tail', np.where(TRAIN_PREV<0.10,'medium','head'))

def stratified(split, min_pos=MIN_POS, n_bg=N_BACKGROUND, seed=SEED):
    sub = ehr[ehr.split==split]; rng = np.random.default_rng(seed); keep = set()
    for l in LABELS:
        pos = sub.index[sub[l]==1].to_numpy()
        keep |= set(rng.choice(pos, min(min_pos, len(pos)), replace=False).tolist())
    rest = np.setdiff1d(sub.index.to_numpy(), np.fromiter(keep, int))
    keep |= set(rng.choice(rest, min(n_bg, len(rest)), replace=False).tolist())
    return np.sort(np.fromiter(keep, int))

ROWS = {s: stratified(s) for s in ('train','val')}
print(f"studies: fit {len(ROWS['train']):,} | eval {len(ROWS['val']):,}")

vm = pd.read_parquet(os.path.join(OUT_DIR,'view_map.parquet'))
vm = vm[vm.view_group.isin(['frontal','lateral'])]
assert 'subject_id' in vm.columns, 'view_map thieu subject_id -> can path Drive'
man = vm[['dicom_id','study_id','subject_id','view_group']].merge(
        ehr[['study_id','split']], on='study_id', how='inner')
SID = {s: set(ehr.loc[ROWS[s],'study_id']) for s in ROWS}
man = man[man.study_id.isin(SID['train'] | SID['val'])].reset_index(drop=True)

def drive_path(sub, st, dic):
    s = str(int(sub)); return os.path.join(DRIVE_IMG, f'p{s[:2]}/p{s}/s{int(st)}/{dic}.jpg')

_r = man.iloc[0]
_p = drive_path(_r.subject_id, _r.study_id, _r.dicom_id)
assert os.path.exists(_p), f'anh goc khong thay: {_p}\n  -> Drive chua mount, hoac 615GB anh khong con o day'
print('anh goc OK, vd:', _p)
IDXm = {s: man.index[man.study_id.isin(SID[s])].to_numpy() for s in ROWS}
print('anh:', {s: int(len(IDXm[s])) for s in IDXm})

studies: fit 5,708 | eval 5,585
anh goc OK, vd: /content/drive/MyDrive/MIMIC_CXR/mimic-cxr-jpg-2.1.0.physionet.org/files/p10/p10125259/s54811596/df653b81-752086fa-d6d268ba-842554bd-e562e8ae.jpg
anh: {'train': 9064, 'val': 8794}


## C2 · Backbone đông cứng + trích 3 crop

`STORE_SIZE = 518×1.08 = 559`. Scale cạnh ngắn tới 559; trục dài dài `L≥559`; crop 559 ở
vị trí `[0, (L-559)//2, L-559]` = `[lo, ctr, hi]`. Mỗi crop → `CenterCrop(518)` + normalize
→ đúng đầu vào ViT. Trích `[CLS ‖ mean(patch)]` = 1536-d mỗi crop.

In [3]:
from transformers import AutoModel, AutoImageProcessor
from torchvision import transforms as T

backbone = AutoModel.from_pretrained('microsoft/rad-dino').to(dev).eval()
_sd = torch.load(BB_CKPT, map_location=dev, weights_only=False)['model']
backbone.load_state_dict({k[9:]: v for k,v in _sd.items() if k.startswith('backbone.')}, strict=True)
for p in backbone.parameters(): p.requires_grad = False
proc = AutoImageProcessor.from_pretrained('microsoft/rad-dino')
MODEL_SIZE = proc.crop_size['height']; MEAN, STD = proc.image_mean[0], proc.image_std[0]
STORE_SIZE = int(round(MODEL_SIZE * CROP_HEADROOM))
to_model = T.Compose([T.CenterCrop(MODEL_SIZE), T.ToTensor(), T.Normalize([MEAN]*3,[STD]*3)])
print(f'MODEL_SIZE {MODEL_SIZE} | STORE_SIZE {STORE_SIZE}')

def three_crops(path):
    """scale canh ngan -> STORE_SIZE, crop STORE_SIZE o 3 vi tri doc truc dai.
    Tra (list[3] tensor da normalize, crop_frac) — crop_frac = phan truc dai bi 04a cat."""
    im = Image.open(path).convert('L')
    w, h = im.size; s = STORE_SIZE / min(w, h)
    im = im.resize((max(1,round(w*s)), max(1,round(h*s))), Image.BICUBIC)
    W, H = im.size
    crops = []
    if H >= W:                                   # portrait: truc dai = doc, shift theo top
        L = H; poss = [0, (L-STORE_SIZE)//2, L-STORE_SIZE]
        boxes = [(0, p, STORE_SIZE, p+STORE_SIZE) for p in poss]
    else:                                        # landscape: truc dai = ngang, shift theo left
        L = W; poss = [0, (L-STORE_SIZE)//2, L-STORE_SIZE]
        boxes = [(p, 0, p+STORE_SIZE, p+STORE_SIZE) for p in poss]
    for b in boxes:
        c = im.crop(b); crops.append(to_model(Image.merge('RGB',(c,c,c))))
    crop_frac = (L - STORE_SIZE) / L             # phan truc dai nam ngoai crop giua
    return crops, float(crop_frac)

@torch.no_grad()
def extract(idx, bs=48):
    """Tra Z[n,3,1536] (lo,ctr,hi) + crop_frac[n]. Doc anh goc tuan tu (Drive random-access)."""
    n = len(idx); Z = np.zeros((n,3,1536),'float32'); CF = np.zeros(n,'float32')
    buf, meta_i = [], []
    def flush():
        if not buf: return
        x = torch.stack(buf).to(dev)                                   # [B*3,3,H,W]
        with torch.autocast('cuda', dtype=torch.bfloat16, enabled=(dev=='cuda')):
            h = backbone(pixel_values=x).last_hidden_state
        z = torch.cat([h[:,0], h[:,1:].mean(1)], -1).float().cpu().numpy()   # [B*3,1536]
        for j,(row_k) in enumerate(meta_i):
            Z[row_k] = z[3*j:3*j+3]
        buf.clear(); meta_i.clear()
    for k, i in enumerate(tqdm(idx, desc='3-crop extract')):
        r = man.iloc[i]
        crops, cf = three_crops(drive_path(r.subject_id, r.study_id, r.dicom_id))
        buf.extend(crops); meta_i.append(k); CF[k] = cf
        if len(meta_i) >= bs: flush()
    flush()
    return Z, CF

FE = {}
for s in ('train','val'):
    f = f'{CS_DIR}/feat3_{s}.npz'
    if os.path.exists(f):
        z = np.load(f); FE[s] = (z['Z'], z['CF'])
        if len(z['Z']) != len(IDXm[s]):
            print(f'{s}: cache lech ({len(z["Z"])} vs {len(IDXm[s])}) -> trich lai'); os.remove(f)
    if s not in FE or len(FE[s][0]) != len(IDXm[s]):
        FE[s] = extract(IDXm[s]); np.savez(f, Z=FE[s][0], CF=FE[s][1])
    print(f'  {s}: Z {FE[s][0].shape} | crop_frac p50 {np.median(FE[s][1]):.1%} p90 {np.percentile(FE[s][1],90):.1%}')

config.json:   0%|          | 0.00/879 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  346MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/223 [00:00<?, ?it/s]

preprocessor_config.json:   0%|          | 0.00/756 [00:00<?, ?B/s]

MODEL_SIZE 518 | STORE_SIZE 559


3-crop extract:   0%|          | 0/9064 [00:00<?, ?it/s]

  train: Z (9064, 3, 1536) | crop_frac p50 16.8% p90 16.8%


3-crop extract:   0%|          | 0/8794 [00:00<?, ?it/s]

  val: Z (8794, 3, 1536) | crop_frac p50 16.8% p90 16.8%


## C3 · Probe — `ctr` vs `mean(3)` vs `concat(3)`, mức study 7:3

In [4]:
from sklearn.metrics import average_precision_score
Y_ST = ehr.set_index('study_id')

def pool_eval(p_img, idx):
    sub = man.iloc[idx]
    d = pd.DataFrame(p_img, columns=LABELS)
    d['study_id']=sub.study_id.to_numpy(); d['vg']=sub.view_group.to_numpy()
    sid = np.sort(d.study_id.unique())
    fr = d[d.vg=='frontal'].groupby('study_id')[LABELS].mean().reindex(sid)
    la = d[d.vg=='lateral'].groupby('study_id')[LABELS].mean().reindex(sid)
    hf = fr.notna().all(1).to_numpy('float32'); hl = la.notna().all(1).to_numpy('float32')
    den = (W_F*hf + W_L*hl)[:,None]
    PS = (W_F*np.nan_to_num(fr.to_numpy('float32'))*hf[:,None]
          + W_L*np.nan_to_num(la.to_numpy('float32'))*hl[:,None]) / den
    Yt = Y_ST.loc[sid, LABELS].to_numpy('float32')
    return np.array([average_precision_score(Yt[:,j], PS[:,j]) for j in range(len(LABELS))])

def feats(mode):
    """mode: 'ctr' | 'mean' | 'concat' -> (Xtr, Xva) da chuan hoa theo fit-set."""
    def build(split):
        Z = FE[split][0]                          # [n,3,1536] = lo,ctr,hi
        if   mode == 'ctr':    X = Z[:,1]
        elif mode == 'mean':   X = Z.mean(1)
        elif mode == 'concat': X = Z.reshape(len(Z), -1)
        return X.astype('float32')
    return build('train'), build('val')

def probe(mode, steps=600, lr=3e-3, wd=1e-4):
    Xtr, Xva = feats(mode)
    Xtr = torch.tensor(Xtr, device=dev); Xva = torch.tensor(Xva, device=dev)
    mu, sd = Xtr.mean(0,keepdim=True), Xtr.std(0,keepdim=True)+1e-6
    Xtr, Xva = (Xtr-mu)/sd, (Xva-mu)/sd
    Ytr = torch.tensor(ehr.set_index('study_id').loc[man.iloc[IDXm['train']].study_id, LABELS]
                       .to_numpy('float32'), device=dev)
    lin = nn.Linear(Xtr.shape[1], len(LABELS)).to(dev)
    opt = torch.optim.AdamW(lin.parameters(), lr=lr, weight_decay=wd)
    pw = ((len(Ytr)-Ytr.sum(0))/Ytr.sum(0).clamp(min=1)).clamp(max=50)
    for _ in range(steps):
        opt.zero_grad(); F.binary_cross_entropy_with_logits(lin(Xtr), Ytr, pos_weight=pw).backward(); opt.step()
    with torch.no_grad(): P = torch.sigmoid(lin(Xva)).cpu().numpy()
    return pool_eval(P, IDXm['val'])

AP = {m: probe(m) for m in ('ctr','mean','concat')}
tab = pd.DataFrame({m: dict(macro=AP[m].mean(), head=AP[m][TIER=='head'].mean(),
                            medium=AP[m][TIER=='medium'].mean(), tail=AP[m][TIER=='tail'].mean())
                    for m in AP}).T.round(4)
print(tab.to_string())
print(f"\nmean  - ctr : {AP['mean'].mean()-AP['ctr'].mean():+.4f}")
print(f"concat- ctr : {AP['concat'].mean()-AP['ctr'].mean():+.4f}")

         macro    head  medium    tail
ctr     0.3867  0.5272  0.2317  0.1782
mean    0.3875  0.5291  0.2306  0.1815
concat  0.3742  0.5074  0.2278  0.1733

mean  - ctr : +0.0008
concat- ctr : -0.0125


## C4 · Phân tầng theo lượng cắt — hiệu ứng phải tập trung ở ảnh bị cắt nhiều

Nếu vùng bị cắt có tín hiệu, lợi thế của `mean/concat` phải **lớn hơn trên study có crop_frac
cao**. Đánh giá lại per-study AP nhưng chỉ trên study mà **mọi ảnh** đều bị cắt nhiều.

In [5]:
# crop_frac o muc study = trung binh cac anh cua study
sub_va = man.iloc[IDXm['val']].copy(); sub_va['cf'] = FE['val'][1]
stud_cf = sub_va.groupby('study_id').cf.mean()

def ap_on(studies, mode):
    idx = IDXm['val'][np.isin(man.iloc[IDXm['val']].study_id.to_numpy(), studies)]
    if len(idx) < 50: return np.nan
    Xtr, _ = feats(mode)
    # tai dung probe da fit tren toan train, chi doi tap eval
    Xtr = torch.tensor(Xtr, device=dev)
    mu, sd = Xtr.mean(0,keepdim=True), Xtr.std(0,keepdim=True)+1e-6
    Xva_all = feats(mode)[1]
    sel = np.isin(man.iloc[IDXm['val']].study_id.to_numpy(), studies)
    Xva = (torch.tensor(Xva_all[sel], device=dev)-mu)/sd
    Ytr = torch.tensor(ehr.set_index('study_id').loc[man.iloc[IDXm['train']].study_id, LABELS].to_numpy('float32'), device=dev)
    lin = nn.Linear(Xtr.shape[1], len(LABELS)).to(dev)
    opt = torch.optim.AdamW(lin.parameters(), lr=3e-3, weight_decay=1e-4)
    Xtr_n = (Xtr-mu)/sd; pw = ((len(Ytr)-Ytr.sum(0))/Ytr.sum(0).clamp(min=1)).clamp(max=50)
    for _ in range(600):
        opt.zero_grad(); F.binary_cross_entropy_with_logits(lin(Xtr_n), Ytr, pos_weight=pw).backward(); opt.step()
    with torch.no_grad(): P = torch.sigmoid(lin(Xva)).cpu().numpy()
    return pool_eval(P, IDXm['val'][sel]).mean()

q = stud_cf.quantile([0, .5, 1.0]).values
lo_stud = stud_cf[stud_cf <= q[1]].index.to_numpy()      # cat it
hi_stud = stud_cf[stud_cf >  q[1]].index.to_numpy()      # cat nhieu
print(f'nguong crop_frac giua = {q[1]:.1%}  | study cat-it {len(lo_stud)} | cat-nhieu {len(hi_stud)}')
for name, st in (('cat-IT ', lo_stud), ('cat-NHIEU', hi_stud)):
    dc = ap_on(st,'ctr'); dm = ap_on(st,'mean')
    print(f'  {name}: ctr {dc:.4f} | mean {dm:.4f} | mean-ctr {dm-dc:+.4f}')

print('\n===== KET LUAN =====')
d_mean = AP['mean'].mean()-AP['ctr'].mean()
if d_mean > 0.005:
    print(f'  mean-ctr = {d_mean:+.4f} > +0.005 -> vung bi cat CO tin hieu.')
    print('  -> dang kiem bang ML-Decoder + 3 seed truoc khi doi 04a (chay lai pipeline).')
else:
    print(f'  mean-ctr = {d_mean:+.4f} <= +0.005 -> vung bi cat KHONG them tin hieu dang ke.')
    print('  -> center-crop cua 04a khong lam mat gi do duoc. Khong doi pipeline.')
json.dump({m: AP[m].tolist() for m in AP}, open(f'{CS_DIR}/crop_ap.json','w'), indent=1)

nguong crop_frac giua = 16.8%  | study cat-it 5130 | cat-nhieu 455
  cat-IT : ctr 0.3886 | mean 0.3889 | mean-ctr +0.0003
  cat-NHIEU: ctr 0.3840 | mean 0.3893 | mean-ctr +0.0053

===== KET LUAN =====
  mean-ctr = +0.0008 <= +0.005 -> vung bi cat KHONG them tin hieu dang ke.
  -> center-crop cua 04a khong lam mat gi do duoc. Khong doi pipeline.
